# Verify that a QNAasm circuit can be executed in nearest neighbors

Inside this notebook, we will see how we can use the verify module to check that a given QNAasm circuit can be executed in nearest neighbors.

Here, "nearest neighbors" means that qubits can interact one with each other if they are direct neighbors on the QPU 2D grid.

Firstly, we need a circuit to check. Lets start with a valid one.

In [1]:
from qnaasm.nodes import (
    Block,
    Calloc,
    Conditional,
    Gate,
    Measure,
    Move,
    Qalloc,
    Qfree,
    QNAasm,
)

from qnaasm.position import Position
from qnaasm.classical_register import ClassicalRegister

In [2]:
instructions = [
    Calloc("c1", 2),
    Calloc("c2", 1),
    Qalloc([Position(0, 0)]),
    Block(
        [
            Move(Position(0, 0), Position(1, 0)),
            Move(Position(1, 0), Position(2, 0)),
        ]
    ),
    Gate("H", [Position(2, 0)]),
    Qalloc([Position(2, 2)]),
    Move(Position(2, 2), Position(2, 1)),
    Gate("CX", [Position(2, 0), Position(2, 1)]),
    Measure([Position(2, 0), Position(2, 1)], ClassicalRegister("c1")),
    Qfree([Position(2, 0)]),
    Conditional(
        ClassicalRegister("c1"),
        3,
        Block(
            [
                Qalloc([Position(0, 2)]),
                Gate("H", [Position(0, 2)]),
                Measure([Position(0, 2)], ClassicalRegister("c2")),
                Qfree([Position(0, 2)]),
            ]
        ),
    ),
    Qfree([Position(2, 1)]),
]

To check that this circuit is valid, we can call the `verify` function.

In [3]:
from verify.verification import verify

def verify_wrapper(instructions: list[QNAasm]):
    if verify(instructions):
        print("Circuit is valid!")
    
    else:
        print("Circuit is invalid.")


verify_wrapper(instructions)

Circuit is valid!


Lets introduce some errors inside this circuit.

In [4]:
instructions_invalid = [
    Calloc("c1", 2),
    Calloc("c2", 1),
    Qalloc([Position(0, 0)]),
    Block(
        [
            Move(Position(0, 0), Position(1, 0)),
            Move(Position(1, 0), Position(2, 0)),
        ]
    ),
    Gate("H", [Position(2, 0)]),
    Qalloc([Position(2, 2)]),
    # Do not move qubit at {2,2} to {2,1}.
    # Move(Position(2, 2), Position(2, 1)),
    Gate("CX", [Position(2, 0), Position(2, 1)]),
    Measure([Position(2, 0), Position(2, 1)], ClassicalRegister("c1")),
    Qfree([Position(2, 0)]),
    Conditional(
        ClassicalRegister("c1"),
        3,
        Block(
            [
                Qalloc([Position(0, 2)]),
                Gate("H", [Position(0, 2), Position(2, 1)]),
                Measure([Position(0, 2)], ClassicalRegister("c1")), # Invalid register size.
                Qfree([Position(0, 2)]),
            ]
        ),
    ),
    Qfree([Position(2, 1)]),
]

In [5]:
verify_wrapper(instructions_invalid)

Circuit is invalid.


target {2,1} is not present on the grid
target {2,1} is not present on the grid
target {2,1} is not present on the grid
target {2,1} is not present on the grid
all qubits have not been deallocated at the end of the circuit
